# 06 · Análisis geoespacial interactivo con Folium

**Objetivo:** responder con mapas tres preguntas:

1. ¿Dónde están los sitios de lanzamiento? ¿Están cerca del ecuador o de la costa?
2. ¿Qué sitios acumulan más aterrizajes exitosos?
3. ¿A qué distancia están de la costa, del ferrocarril, de las autopistas y de las ciudades?

Los mapas se guardan como HTML interactivo en la carpeta `maps/`.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
IMAGES_DIR = Path("../images")
for folder in (DATA_DIR, RESULTS_DIR, IMAGES_DIR):
    folder.mkdir(exist_ok=True)

# Repositorio público de datos del curso IBM Applied Data Science Capstone
IBM = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"


def load_csv(name: str, url: str) -> pd.DataFrame:
    """Lee data/<name> si ya existe; si no, lo descarga desde `url` y lo guarda como caché local."""
    local = DATA_DIR / name
    if local.exists():
        print(f"Leyendo {local}")
        return pd.read_csv(local)
    print(f"Descargando {url}")
    df = pd.read_csv(url)
    df.to_csv(local, index=False)
    return df

In [ ]:
import json
from math import asin, cos, radians, sin, sqrt

import folium
from folium.features import DivIcon
from folium.plugins import MarkerCluster, MousePosition

MAPS_DIR = Path("../maps")
MAPS_DIR.mkdir(exist_ok=True)

spacex_df = load_csv("spacex_launch_geo.csv", f"{IBM}/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[["Launch Site", "Lat", "Long", "class"]]
launch_sites_df = spacex_df.groupby("Launch Site", as_index=False).first()[["Launch Site", "Lat", "Long"]]
launch_sites_df

## Tarea 1 · Marcar todos los sitios de lanzamiento

In [ ]:
NASA_JSC = [29.559684888503615, -95.0830971930759]  # Centro Espacial Johnson (Houston), referencia

site_map = folium.Map(location=[29.0, -89.0], zoom_start=5)
folium.Circle(NASA_JSC, radius=1000, color="#d35400", fill=True).add_child(
    folium.Popup("NASA Johnson Space Center")).add_to(site_map)

for _, site in launch_sites_df.iterrows():
    coordinate = [site["Lat"], site["Long"]]
    folium.Circle(coordinate, radius=1000, color="#000000", fill=True).add_child(
        folium.Popup(site["Launch Site"])).add_to(site_map)
    folium.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html=f'<div style="font-size: 12px; color:#d35400;"><b>{site["Launch Site"]}</b></div>'),
    ).add_to(site_map)

site_map.save(MAPS_DIR / "01_launch_sites.html")
site_map

**Lectura:** los cuatro sitios están en la costa (tres en Florida, uno en California) y lo más al sur
posible dentro de EE. UU. continental: cuanto más cerca del ecuador, más ayuda la rotación terrestre.

## Tarea 2 · Éxitos y fallos por sitio con `MarkerCluster`

In [ ]:
spacex_df["marker_color"] = spacex_df["class"].map({1: "green", 0: "red"})

outcome_map = folium.Map(location=[29.0, -89.0], zoom_start=5)
marker_cluster = MarkerCluster().add_to(outcome_map)
for _, record in spacex_df.iterrows():
    folium.Marker(
        location=[record["Lat"], record["Long"]],
        icon=folium.Icon(color="white", icon_color=record["marker_color"]),
        popup=f'{record["Launch Site"]} · {"Éxito" if record["class"] == 1 else "Fallo"}',
    ).add_to(marker_cluster)

outcome_map.save(MAPS_DIR / "02_launch_outcomes.html")
outcome_map

In [ ]:
site_outcomes = spacex_df.groupby("Launch Site")["class"].agg(total="count", exitos="sum")
site_outcomes["tasa_exito"] = (site_outcomes["exitos"] / site_outcomes["total"]).round(4)
site_outcomes.sort_values("tasa_exito", ascending=False)

## Tarea 3 · Distancias desde CCAFS SLC-40 a su entorno

In [ ]:
def calculate_distance(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos coordenadas (fórmula del haversine)."""
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    return 2 * R * asin(sqrt(a))


site_name = "CCAFS SLC-40"
site_row = launch_sites_df[launch_sites_df["Launch Site"] == site_name].iloc[0]
launch_site = (site_row["Lat"], site_row["Long"])

# Puntos de referencia leídos en el mapa con MousePosition (coordenadas aproximadas).
points_of_interest = {
    "Costa (playa)": (28.56342, -80.56794),
    "Ferrocarril (NASA Railroad)": (28.57206, -80.58525),
    "Autopista (Samuel C. Phillips Pkwy)": (28.56276, -80.58700),
    "Ciudad (Titusville)": (28.61200, -80.80788),
}
distances = {
    name: round(calculate_distance(*launch_site, *coords), 2) for name, coords in points_of_interest.items()
}
pd.Series(distances, name="distancia_km")

In [ ]:
proximity_map = folium.Map(location=list(launch_site), zoom_start=13)
proximity_map.add_child(MousePosition(position="topright", separator=" Long: ", prefix="Lat:",
                                      num_digits=5, empty_string="NaN", lat_formatter=None, lng_formatter=None))
folium.Marker(list(launch_site), popup=site_name, icon=folium.Icon(color="orange", icon="rocket", prefix="fa")
              ).add_to(proximity_map)

line_colors = ["#1f77b4", "#2ca02c", "#9467bd", "#d62728"]
for (name, coords), color in zip(points_of_interest.items(), line_colors):
    folium.Marker(
        coords,
        icon=DivIcon(icon_size=(220, 20), icon_anchor=(0, 0),
                     html=f'<div style="font-size: 12px; color:{color};"><b>{name}: {distances[name]:.2f} km</b></div>'),
    ).add_to(proximity_map)
    folium.PolyLine(locations=[list(launch_site), list(coords)], weight=3, color=color).add_to(proximity_map)

proximity_map.save(MAPS_DIR / "03_proximities.html")
proximity_map

In [ ]:
summary = {
    "launch_sites": launch_sites_df.to_dict(orient="records"),
    "site_outcomes": site_outcomes.reset_index().to_dict(orient="records"),
    "proximities_site": site_name,
    "proximities_km": distances,
}
(RESULTS_DIR / "folium.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False))
summary

## Conclusiones geoespaciales

* **Costa:** los sitios están a menos de 1–2 km del mar; así, el ascenso y un posible aborto ocurren sobre
  el océano, lejos de personas.
* **Infraestructura:** ferrocarril y autopistas cercanos facilitan el transporte de piezas enormes.
* **Ciudades:** se mantienen a decenas de kilómetros por seguridad.
* **Latitud:** Florida está más cerca del ecuador, lo que da un "impulso" extra hacia órbitas ecuatoriales;
  Vandenberg (California) se usa para órbitas polares lanzando hacia el sur sobre el Pacífico.